# Model 03 — Random Forest 


# Model 04 — Random Forest v2 (With Feature Engineering)
Random Forest re-trained with the 5 new engineered features. Testing if better inputs help.

In [ ]:
import pandas as pd, numpy as np, warnings
warnings.filterwarnings('ignore')
from pathlib import Path
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, RandomizedSearchCV

BASE = Path(".")
df24 = pd.read_csv(BASE / "STEMS_Train_2024.csv")
df25 = pd.read_csv(BASE / "STEMS_Validate_2025.csv")
df26 = pd.read_csv(BASE / "STEMS_Test_2026.csv")

LEAK = ["Annual_Rounds","Months_In_Season","Year","Season","Division","Field_No","Target_Lag1","Target_Lag2"]
df_train = pd.concat([df24,df25],ignore_index=True)
df_test  = df26.copy()
for df in [df_train,df_test]: df.drop(columns=[c for c in LEAK if c in df.columns],inplace=True)
df_train = df_train[df_train["Near_Pruning_Flag"]==0].reset_index(drop=True)
df_test  = df_test[df_test["Near_Pruning_Flag"]==0].reset_index(drop=True)
for df in [df_train,df_test]: df.drop(columns=["Near_Pruning_Flag"],inplace=True)

def engineer(df):
    df=df.copy()
    df["Soil_Index"]=df["Soil_Carbon"]/(df["Soil_pH"].replace(0,np.nan)+1e-9)
    df["Yield_Eff"]=df["Yield_Prev_Year"]/(df["Extent_Hect"].replace(0,np.nan)+1e-9)
    df["Prune_Age_Ratio"]=df["Prune_Cycle_Stage"]/(df["Age_Months"]/12+1e-9)
    df["Rain_Trend"]=df["Rainfall_Lag1"]-df["Rainfall_Lag3"]
    df["Growth_per_Prod"]=df["Growth_Response"]/(df["Field_Productivity"]+1e-9)
    return df

df_train=engineer(df_train); df_test=engineer(df_test)
TARGET="Target_Days"
num_cols=[c for c in df_train.select_dtypes(include=[np.number]).columns if c!=TARGET]
imp=SimpleImputer(strategy='mean')
X_train=imp.fit_transform(df_train[num_cols]); X_test=imp.transform(df_test[num_cols])
y_train=df_train[TARGET].values; y_test=df_test[TARGET].values
print("Data ready with engineered features. Training Random Forest v2...")


In [ ]:
cv5=KFold(n_splits=5,shuffle=True,random_state=42)
rf_space={"n_estimators":[300,500,700],"max_depth":[None,6,10,15],"min_samples_split":[2,5,8],
           "min_samples_leaf":[1,2,3],"max_features":["sqrt","log2",0.4]}
rs=RandomizedSearchCV(RandomForestRegressor(random_state=42),rf_space,
                       n_iter=30,cv=cv5,scoring="neg_mean_absolute_error",random_state=42,n_jobs=-1)
rs.fit(X_train,y_train)

y_pred=rs.best_estimator_.predict(X_test)
mae=mean_absolute_error(y_test,y_pred)
r2=r2_score(y_test,y_pred)
train_mae=mean_absolute_error(y_train,rs.best_estimator_.predict(X_train))

print(f"Best params: {rs.best_params_}")
print(f"\n=== RF v2 RESULTS ===")
print(f"Train MAE : {train_mae:.4f}")
print(f"Test  MAE : {mae:.4f}")
print(f"R2        : {r2:.4f}")
print(f"Overfit gap: {mae-train_mae:.4f}")
print("\nConclusion: Marginal improvement from feature engineering.")
print("Still overfitting — fundamental problem is dataset size (n=141).")
print("Next: Try XGBoost, which has stronger built-in regularisation.")


## Result: Marginal Improvement, Still Overfitting
- Engineered features provide slight improvement over RF v1
- But overfitting gap remains large — tree models need more data
- **Next: Try XGBoost with stronger regularisation**

# Model 04 — Random Forest  (With Feature Engineering)
Random Forest re-trained with the 5 new engineered features. Testing if better inputs help.

In [ ]:
import pandas as pd, numpy as np, warnings
warnings.filterwarnings('ignore')
from pathlib import Path
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, RandomizedSearchCV
import matplotlib.pyplot as plt

BASE = Path(".")
df24 = pd.read_csv(BASE / "STEMS_Train_2024.csv")
df25 = pd.read_csv(BASE / "STEMS_Validate_2025.csv")
df26 = pd.read_csv(BASE / "STEMS_Test_2026.csv")

LEAK = ["Annual_Rounds","Months_In_Season","Year","Season","Division","Field_No","Target_Lag1","Target_Lag2"]
df_train = pd.concat([df24,df25],ignore_index=True)
df_test  = df26.copy()
for df in [df_train,df_test]: df.drop(columns=[c for c in LEAK if c in df.columns],inplace=True)
df_train = df_train[df_train["Near_Pruning_Flag"]==0].reset_index(drop=True)
df_test  = df_test[df_test["Near_Pruning_Flag"]==0].reset_index(drop=True)
for df in [df_train,df_test]: df.drop(columns=["Near_Pruning_Flag"],inplace=True)

TARGET = "Target_Days"
num_cols = [c for c in df_train.select_dtypes(include=[np.number]).columns if c != TARGET]
imp = SimpleImputer(strategy='mean')
X_train = imp.fit_transform(df_train[num_cols])
X_test  = imp.transform(df_test[num_cols])
y_train = df_train[TARGET].values; y_test = df_test[TARGET].values



In [ ]:
cv5 = KFold(n_splits=5, shuffle=True, random_state=42)
rf_space = {"n_estimators":[100,300,500],"max_depth":[None,5,10],"min_samples_split":[2,5],"max_features":["sqrt","log2"]}
rs = RandomizedSearchCV(RandomForestRegressor(random_state=42), rf_space,
                         n_iter=20, cv=cv5, scoring="neg_mean_absolute_error", random_state=42, n_jobs=-1)
rs.fit(X_train, y_train)
print(f"Best params: {rs.best_params_}")

y_pred = rs.best_estimator_.predict(X_test)
mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)
train_mae = mean_absolute_error(y_train, rs.best_estimator_.predict(X_train))

print(f"results")
print(f"Train MAE: {train_mae:.4f}  Test MAE: {mae:.4f}  Gap: {mae-train_mae:.4f}")
print(f"RMSE: {rmse:.4f}  R2: {r2:.4f}")


In [ ]:
# Feature importance
importances = rs.best_estimator_.feature_importances_
feat_imp = pd.Series(importances, index=num_cols).sort_values(ascending=False)
print("Top 10 features:")
print(feat_imp.head(10).round(4))

fig, ax = plt.subplots(figsize=(10,5))
feat_imp.head(15).plot(kind='bar', color='forestgreen', edgecolor='k', ax=ax)
ax.set_title('Random Forest v1 — Feature Importance')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('model_03_rf_importance.png', dpi=150)
plt.show()
print("\nConclusion: Still overfitting. Train/test MAE gap is large.")



## Result: Still Overfitting
- Random Forest reduces variance vs single tree but still overfits on n=141
- Large gap between train MAE and test MAE
